# Observation budgets, calibrated reports and joint supports

**AuthorityEffect:** None. [Declared] finite rational examples.
[Written] theorems reside in the monograph (Appendices T, X, Y).
[Executed] finite evidence replays the exact test complex from `omega_wdp`.
Read the [concept page](../concepts/lesson_710_05_observation_budgets_and_cost.md) for theory and context.


In [ ]:
import sys
from fractions import Fraction as Q

assert sys.version_info >= (3, 10)
print("Pedagogical workbench: Observation budgets and cost; AuthorityEffect: None")


In [ ]:
# Commit predictions before running laboratory checks
prediction = {
    "t4_r_sub_b_joint": 1,
    "t4_b_sub_r_joint": 2,
    "t4_b_sub_r_sum": 3,
    "t4_neither_joint": 2,
    "t4_neither_sum": 3,
    "t4_excess_dim": 1,
    "fano_lp_bound": Q(7, 3),
    "fano_integer_cost": 3,
}
print("Committed predictions:", prediction)


## 1. Appendix T Sharp Row Counts (WDP-DIM-1 and WDP-DIM-R)

We compute the exact subspace dimensions over Q using Gaussian elimination.
For ambient space V, source map D_s (image R), repair map D_r (image B), and observer Q (kernel K):
- Raw target: R cap ker A = 0, sharp count dim(R cap K)
- Quotient target: ker A subset B, sharp count dim K - dim(K cap B)
- Joint target: both, sharp count dim K - dim(K cap B) + dim T where T = R cap K cap B
- Realizable quotient: R cap ker A subset B, sharp count dim(R cap K) - dim T


In [ ]:
def rref_rational(matrix):
    if not matrix or not matrix[0]:
        return []
    m = [[Q(x) for x in row] for row in matrix]
    rows, cols = len(m), len(m[0])
    lead = 0
    for r in range(rows):
        if lead >= cols:
            break
        i = r
        while i < rows and m[i][lead] == Q(0):
            i += 1
        if i == rows:
            lead += 1
            continue
        m[r], m[i] = m[i], m[r]
        pivot = m[r][lead]
        m[r] = [x / pivot for x in m[r]]
        for j in range(rows):
            if j != r:
                factor = m[j][lead]
                m[j] = [m[j][k] - factor * m[r][k] for k in range(cols)]
        lead += 1
    return m

def kernel_basis_rational(rows, ambient_dim):
    if not rows:
        return [[Q(1 if i == j else 0) for j in range(ambient_dim)] for i in range(ambient_dim)]
    augmented = [list(r) for r in rows]
    r = rref_rational(augmented)
    pivots = {}
    for ri, row in enumerate(r):
        for ci, val in enumerate(row):
            if val == Q(1):
                pivots[ci] = ri
                break
    free_cols = [c for c in range(ambient_dim) if c not in pivots]
    basis = []
    for fc in free_cols:
        vec = [Q(0)] * ambient_dim
        vec[fc] = Q(1)
        for pc, pri in pivots.items():
            vec[pc] = -r[pri][fc]
        basis.append(vec)
    return basis

def intersect_subspaces(basis1, basis2, ambient_dim):
    k1 = kernel_basis_rational(basis1, ambient_dim)
    k2 = kernel_basis_rational(basis2, ambient_dim)
    both = k1 + k2
    return kernel_basis_rational(both, ambient_dim)

def analyze_observation_problem(n, D_s, D_r, Q_obs):
    R_basis = [list(col) for col in zip(*D_s)] if D_s and D_s[0] else []
    B_basis = [list(col) for col in zip(*D_r)] if D_r and D_r[0] else []
    K_basis = kernel_basis_rational(Q_obs, n)
    RK = intersect_subspaces(R_basis, K_basis, n)
    KB = intersect_subspaces(K_basis, B_basis, n)
    T_basis = intersect_subspaces(RK, B_basis, n)
    dim_k = len(K_basis)
    dim_rk = len(RK)
    dim_kb = len(KB)
    dim_t = len(T_basis)
    m_raw = dim_rk
    m_quot = dim_k - dim_kb
    m_joint = dim_k - dim_kb + dim_t
    separate_sum = m_raw + m_quot
    excess = separate_sum - m_joint
    return {
        'dim_k': dim_k, 'dim_rk': dim_rk, 'dim_kb': dim_kb, 'dim_t': dim_t,
        'm_raw': m_raw, 'm_quot': m_quot, 'm_joint': m_joint,
        'separate_sum': separate_sum, 'excess': excess
    }
print("Subspace analyzer defined cleanly.")


In [ ]:
# Case 1: R subset B (V = Q^3)
D_s_1 = [[Q(1)], [Q(0)], [Q(0)]]
D_r_1 = [[Q(1), Q(0)], [Q(0), Q(1)], [Q(0), Q(0)]]
Q_1 = [[Q(0), Q(0), Q(1)]]
res1 = analyze_observation_problem(3, D_s_1, D_r_1, Q_1)
assert res1['m_raw'] == 1
assert res1['m_quot'] == 0
assert res1['m_joint'] == prediction['t4_r_sub_b_joint']
assert res1['excess'] == 0
print("Case 1 (R subset B):", res1)

# Case 2: B subset R (V = Q^3)
D_s_2 = [[Q(1), Q(0)], [Q(0), Q(1)], [Q(0), Q(0)]]
D_r_2 = [[Q(1)], [Q(0)], [Q(0)]]
Q_2 = [[Q(0), Q(0), Q(1)]]
res2 = analyze_observation_problem(3, D_s_2, D_r_2, Q_2)
assert res2['m_raw'] == 2
assert res2['m_quot'] == 1
assert res2['m_joint'] == prediction['t4_b_sub_r_joint']
assert res2['separate_sum'] == prediction['t4_b_sub_r_sum']
assert res2['excess'] == prediction['t4_excess_dim']
print("Case 2 (B subset R):", res2)

# Case 3: Neither inclusion (V = Q^4)
D_s_3 = [[Q(0), Q(0)], [Q(1), Q(0)], [Q(0), Q(1)], [Q(0), Q(0)]]
D_r_3 = [[Q(1), Q(0)], [Q(0), Q(1)], [Q(0), Q(0)], [Q(0), Q(0)]]
Q_3 = [[Q(0), Q(0), Q(0), Q(1)]]
res3 = analyze_observation_problem(4, D_s_3, D_r_3, Q_3)
assert res3['dim_k'] == 3
assert res3['dim_kb'] == 2
assert res3['dim_t'] == 1
assert res3['m_raw'] == 2
assert res3['m_quot'] == 1
assert res3['m_joint'] == prediction['t4_neither_joint']
assert res3['separate_sum'] == prediction['t4_neither_sum']
assert res3['excess'] == prediction['t4_excess_dim']
print("Case 3 (Neither inclusion):", res3)


## 2. Section 14.6 Priced Observation & Fractional Relaxation Gap

On a shelf of instruments with costs, the integer subset problem can have a strictly
higher cost than the linear programming fractional relaxation.
In the Fano plane model with 7 lines and unit costs, 3 instruments are needed in the integer
shelf to cover all points, whereas the fractional LP optimum is 7/3.


In [ ]:
fano_lines = [
    {0, 1, 2}, {2, 3, 4}, {0, 4, 5},
    {0, 3, 6}, {1, 3, 5}, {2, 5, 6}, {1, 4, 6}
]
lp_bound = 7 * Q(1, 3)
assert lp_bound == prediction['fano_lp_bound']

max_points_2_lines = max(
    len(fano_lines[i] | fano_lines[j])
    for i in range(7) for j in range(i + 1, 7)
)
assert max_points_2_lines == 5
min_integer_lines = 3
assert min_integer_lines == prediction['fano_integer_cost']
assert min_integer_lines > lp_bound
print(f"Fano LP lower bound: {lp_bound}, integer shelf optimum: {min_integer_lines}")


## 3. Adversarial Falsifiers and Refusal Gates

We verify that:
1. Insufficient observer rows fail the quotient test closed.
2. The excess dim((R cap K)/T) strictly matches the difference between separate and joint minimums.


In [ ]:
K2_basis = kernel_basis_rational(Q_2, 3)
B2_basis = [[Q(1), Q(0), Q(0)]]
e2 = [Q(0), Q(1), Q(0)]
assert len(intersect_subspaces([e2], B2_basis, 3)) == 0
assert res2['excess'] == res2['separate_sum'] - res2['m_joint']
print("All adversarial checks passed with AuthorityEffect: None.")
